[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/edit-state-human-feedback.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239520-lesson-3-editing-state-and-human-feedback)

# 编辑图状态

## 回顾

我们讨论了人工参与循环的动机：

(1) `批准` - 我们可以中断我们的agent，向用户显示状态，并允许用户接受某个行动

(2) `调试` - 我们可以回退图以重现或避免问题

(3) `编辑` - 你可以修改状态

我们展示了断点如何支持用户批准，但还不知道如何在图被中断后修改我们的图状态！

## 目标

现在，让我们展示如何直接编辑图状态并插入人工反馈。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_openai langgraph_sdk langgraph-prebuilt

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

## 编辑状态

之前，我们介绍了断点。

我们使用它们来中断图并等待用户批准，然后再执行下一个节点。

但断点也是[修改图状态的机会](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/edit-graph-state/)。

让我们在`assistant`节点之前设置一个断点来设置我们的agent。

In [ ]:
from langchain_openai import ChatOpenAI

def multiply(a: int, b: int) -> int:
    """将a和b相乘。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a * b

# 这将是一个工具
def add(a: int, b: int) -> int:
    """将a和b相加。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a + b

def divide(a: int, b: int) -> float:
    """将a除以b。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode

from langchain_core.messages import HumanMessage, SystemMessage

# 系统消息
sys_msg = SystemMessage(content="您是一个有用的助手，负责对一组输入执行算术运算。")

# 节点
def assistant(state: MessagesState):
   """助手节点，处理消息并调用LLM"""
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些做实际工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定义边：这些确定控制流
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果来自assistant的最新消息（结果）是工具调用 -> tools_condition路由到tools
    # 如果来自assistant的最新消息（结果）不是工具调用 -> tools_condition路由到END
    tools_condition,
)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
graph = builder.compile(interrupt_before=["assistant"], checkpointer=memory)

# 显示
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

让我们运行！

我们可以看到图在聊天模型响应之前被中断。

In [ ]:
# 输入
initial_input = {"messages": "计算2乘以3"}

# 线程
thread = {"configurable": {"thread_id": "1"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

In [ ]:
state = graph.get_state(thread)
state

现在，我们可以直接应用状态更新。

记住，对`messages`键的更新将使用`add_messages`减速器：

* 如果我们想要覆盖现有消息，我们可以提供消息`id`。
* 如果我们只是想要附加到我们的消息列表，那么我们可以传递一个没有指定`id`的消息，如下所示。

In [ ]:
graph.update_state(
    thread,
    {"messages": [HumanMessage(content="不，实际上计算3乘以3！")]},
)

让我们看一下。

我们用新消息调用了`update_state`。

`add_messages`减速器将其附加到我们的状态键`messages`。

In [ ]:
new_state = graph.get_state(thread).values
for m in new_state['messages']:
    m.pretty_print()

现在，让我们通过简单地传递`None`并允许它从当前状态继续来进行我们的agent。

我们发出当前状态，然后继续执行剩余的节点。

In [ ]:
for event in graph.stream(None, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

现在，我们回到了有`断点`的`assistant`。

我们可以再次传递`None`来继续。

In [ ]:
for event in graph.stream(None, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

### 在Studio中编辑图状态

**⚠️ 免责声明**

自从拍摄这些视频以来，我们更新了Studio，使其可以在本地运行并在浏览器中打开。这现在是运行Studio的首选方式（而不是像视频中显示的那样使用桌面应用程序）。请参阅[这里](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server)的本地开发服务器文档和[这里](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server)的相关说明。要启动本地开发服务器，请在此模块的`/studio`目录中的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到Studio UI：`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`。

LangGraph API [支持编辑图状态](https://langchain-ai.github.io/langgraph/cloud/how-tos/human_in_the_loop_edit_state/#initial-invocation)。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("很抱歉，Google Colab目前不支持LangGraph Studio")

In [ ]:
# 这是本地开发服务器的URL
from langgraph_sdk import get_client
client = get_client(url="http://127.0.0.1:2024")

我们的agent在`studio/agent.py`中定义。

如果你查看代码，你会发现它*没有*断点！

当然，我们可以将其添加到`agent.py`，但API的一个非常好的功能是我们可以传入断点！

在这里，我们传递`interrupt_before=["assistant"]`。

In [ ]:
initial_input = {"messages": "计算2乘以3"}
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    "agent",
    input=initial_input,
    stream_mode="values",
    interrupt_before=["assistant"],
):
    print(f"接收到类型为: {chunk.event}的新事件...")
    messages = chunk.data.get('messages', [])
    if messages:
        print(messages[-1])
    print("-" * 50)

我们可以获取当前状态

In [ ]:
current_state = await client.threads.get_state(thread['thread_id'])
current_state

我们可以查看状态中的最后一条消息。

In [ ]:
last_message = current_state['values']['messages'][-1]
last_message

我们可以编辑它！

In [ ]:
last_message['content'] = "不，实际上计算3乘以3！"
last_message

In [ ]:
last_message

记住，如我们之前所说，对`messages`键的更新将使用相同的`add_messages`减速器。

如果我们想要覆盖现有消息，那么我们可以提供消息`id`。

在这里，我们做了这件事。我们只修改了消息`内容`，如上所示。

In [ ]:
await client.threads.update_state(thread['thread_id'], {"messages": last_message})

现在，我们通过传递`None`来恢复。

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="values",
    interrupt_before=["assistant"],
):
    print(f"接收到类型为: {chunk.event}的新事件...")
    messages = chunk.data.get('messages', [])
    if messages:
        print(messages[-1])
    print("-" * 50)

我们得到工具调用的结果为`9`，如预期。

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="values",
    interrupt_before=["assistant"],
):
    print(f"接收到类型为: {chunk.event}的新事件...")
    messages = chunk.data.get('messages', [])
    if messages:
        print(messages[-1])
    print("-" * 50)

## 等待用户输入

所以，很明显我们可以在断点后编辑我们的agent状态。

现在，如果我们想要允许人工反馈来执行这个状态更新怎么办？

我们将添加一个[作为我们agent内人工反馈占位符](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/wait-user-input/#setup)的节点。

这个`human_feedback`节点允许用户直接将反馈添加到状态。

我们使用`interrupt_before`我们的`human_feedback`节点来指定断点。

我们设置检查点以保存图的状态直到这个节点。

In [ ]:
# 系统消息
sys_msg = SystemMessage(content="您是一个有用的助手，负责对一组输入执行算术运算。")

# 应该被中断的无操作节点
def human_feedback(state: MessagesState):
    """人工反馈节点，用作占位符"""
    pass

# 助手节点
def assistant(state: MessagesState):
   """助手节点，处理消息并调用LLM"""
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些做实际工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_node("human_feedback", human_feedback)

# 定义边：这些确定控制流
builder.add_edge(START, "human_feedback")
builder.add_edge("human_feedback", "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果来自assistant的最新消息（结果）是工具调用 -> tools_condition路由到tools
    # 如果来自assistant的最新消息（结果）不是工具调用 -> tools_condition路由到END
    tools_condition,
)
builder.add_edge("tools", "human_feedback")

memory = MemorySaver()
graph = builder.compile(interrupt_before=["human_feedback"], checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

我们将从用户那里获得反馈。

我们使用`.update_state`来用我们获得的人工响应更新图的状态，如之前一样。

我们使用`as_node="human_feedback"`参数将此状态更新应用为指定节点`human_feedback`。

In [ ]:
# 输入
initial_input = {"messages": "计算2乘以3"}

# 线程
thread = {"configurable": {"thread_id": "5"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()
    
# 获取用户输入
user_input = input("告诉我您想如何更新状态：")

# 我们现在更新状态，就像我们是human_feedback节点一样
graph.update_state(thread, {"messages": user_input}, as_node="human_feedback")

# 继续图执行
for event in graph.stream(None, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()

In [ ]:
# 继续图执行
for event in graph.stream(None, thread, stream_mode="values"):
    event["messages"][-1].pretty_print()